### Wave C — NI remainder

NI = stage `Not Interested`. Reason = history `cancelledreason` → `leadcancellationreasons`, else `reason`.

| Sheet | Yellow box |
|-------|------------|
| `NI_Conversion` | #10 (+ same-day NI % sample cols) |
| `NI_By_Source` | #11 |
| `NI_By_Channel` | #17 |
| `NI_Reasons` | #12, #13, #41 |



In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path("/Users/lekhanigam/data/leads_sv_closure_reports")
sys.path.insert(0, str(ROOT))
from common import START_TS, END_TS, q, write_excel

base = q(f"""
WITH ct AS (
  SELECT CAST(cid AS varchar) cid, MAX(CAST(source AS varchar)) src
  FROM campaigntrackers WHERE cid IS NOT NULL GROUP BY 1
),
ni AS (
  SELECT CAST(h.lead AS varchar) lead_oid,
         MIN(h.createdat) ni_at,
         MAX(CAST(h.cancelledreason AS varchar)) cr,
         MAX(CAST(h.reason AS varchar)) reason_txt
  FROM leadstagehistories h
  WHERE CAST(h.stage AS varchar)='Not Interested'
    AND h.createdat >= {START_TS} AND h.createdat < {END_TS}
  GROUP BY 1
)
SELECT
  CAST(l._id AS varchar) lead_oid,
  l.createdat born_at,
  date_format(l.createdat, '%Y-%m') month_key,
  date_format(l.createdat, '%M') month_name,
  TRIM(CAST(ls.name AS varchar)) source,
  COALESCE(NULLIF(TRIM(CAST(us.name AS varchar)), ''), TRIM(CAST(ls.name AS varchar))) channel,
  ni.ni_at,
  COALESCE(NULLIF(TRIM(CAST(r.name AS varchar)), ''), NULLIF(TRIM(ni.reason_txt), '')) ni_reason
FROM leads l
LEFT JOIN leadsources ls ON CAST(ls._id AS varchar)=CAST(l.bornsource AS varchar)
LEFT JOIN ct ON ct.cid=TRIM(CAST(l.utm_campaign AS varchar))
LEFT JOIN utmsources us ON CAST(us._id AS varchar)=ct.src
LEFT JOIN ni ON ni.lead_oid=CAST(l._id AS varchar)
LEFT JOIN leadcancellationreasons r ON CAST(r._id AS varchar)=ni.cr
WHERE l.createdat >= {START_TS} AND l.createdat < {END_TS}
""")
base["is_ni"] = base["ni_at"].notna().astype(int)
base["same_day_ni"] = (
    base["ni_at"].notna()
    & (pd.to_datetime(base["ni_at"]).dt.date == pd.to_datetime(base["born_at"]).dt.date)
).astype(int)
ni = base[base["is_ni"] == 1]
conv = (
    base.groupby(["month_name", "month_key"], as_index=False)
    .agg(**{"Sum of Total Leads": ("lead_oid", "nunique"), "Sum of Not Interested": ("is_ni", "sum"), "Same-day NI": ("same_day_ni", "sum")})
    .sort_values("month_key")
)
conv["Percent NI Same Day"] = np.where(conv["Sum of Not Interested"] > 0, (100 * conv["Same-day NI"] / conv["Sum of Not Interested"]).round(2), 0)
sheets = {
    "NI_Conversion": conv.drop(columns=["month_key", "Same-day NI"]),
    "NI_By_Source": ni.groupby(["source"], as_index=False).agg(**{"NI count": ("lead_oid", "nunique")}).sort_values("NI count", ascending=False),
    "NI_By_Channel": ni.groupby(["channel"], as_index=False).agg(**{"NI count": ("lead_oid", "nunique")}).sort_values("NI count", ascending=False),
    "NI_Reasons": ni.groupby(["ni_reason"], as_index=False).agg(**{"NI count": ("lead_oid", "nunique")}).sort_values("NI count", ascending=False),
}
write_excel(ROOT / "NI_Remainder_Analysis.xlsx", sheets)
print("NI", int(base.is_ni.sum()))

